<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 2 &middot; Part 2 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Multiple Linear Regression

One feature is never enough: nobody prices a flat on floor area alone. Here the training
set becomes a matrix, $w$ becomes a vector, and you build the cost and the gradient for
$n$ features &mdash; then watch gradient descent struggle with them, which is the problem
Part 3 exists to solve.

**Companion theory lecture:** CSE 815, Week 2, Part 1 (Multiple Features, Vectorization, Gradient Descent in $n$ Dimensions).

## What you will be able to do

1. Store a multi-feature training set as `X` of shape `(m, n)` and read off $x^{(i)}_j$.
2. Implement $f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b$ for one example
   and for the whole set.
3. Implement $J(\mathbf{w},b)$ and $\nabla J$ with a loop and vectorised, and prove they agree.
4. Verify the gradient against finite differences before training with it.
5. Run gradient descent on $n$ features, and check the answer against `np.linalg.lstsq`.

---

## 1. The data set

Sixty flats from Chattogram, four features each. This is the data set for the rest of
Week 2, so read the loader &mdash; you will re-use it in Parts 3 and 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)

FEATURES = ["size (k sq ft)", "bedrooms", "floor", "age (yr)"]


def load_flats(seed=815, m=60):
    """Chattogram flats: four features, price in lakh BDT.

    Returns:
        X (ndarray (m, 4)): size, bedrooms, floor number, age in years
        y (ndarray (m,))  : price in lakh BDT
    """
    rng = np.random.default_rng(seed)
    size  = np.round(rng.uniform(0.60, 3.00, m), 2)
    beds  = np.clip(np.round(1.2 + 1.7 * size + rng.normal(0, 0.6, m)), 1, 6)
    floor = rng.integers(1, 13, m).astype(float)
    age   = np.round(rng.uniform(1, 60, m), 0)
    price = (22.0 * size + 2.5 * beds + 0.9 * floor - 0.35 * age + 8.0
             + rng.normal(0, 2.5, m))
    return np.column_stack([size, beds, floor, age]), np.round(price, 1)


X_train, y_train = load_flats()
m, n = X_train.shape

print("X_train.shape :", X_train.shape, " -> m =", m, "examples, n =", n, "features")
print("y_train.shape :", y_train.shape)

In [ ]:
print(f"{'i':>3}  {'size':>6} {'beds':>5} {'floor':>6} {'age':>5}   {'price':>7}")
print("-" * 42)
for i in range(6):
    s, bd, fl, ag = X_train[i]
    print(f"{i:>3}  {s:6.2f} {bd:5.0f} {fl:6.0f} {ag:5.0f}   {y_train[i]:7.1f}")
print("  ...")

print(f"\nx^(2)   = {X_train[2]}      (one row = one example)")
print(f"x^(2)_1 = {X_train[2, 1]}                       (bedrooms of example 2)")
print(f"feature 3 across all examples: {X_train[:, 3][:8]} ...")

### Look at the ranges before anything else

The columns are measured in completely different units. Note the last column especially: it
is the reason Part 3 exists.

In [ ]:
print(f"{'feature':16s} {'min':>8} {'max':>8} {'mean':>8} {'std':>8} {'range':>8}")
print("-" * 62)
for j in range(n):
    col = X_train[:, j]
    print(f"{FEATURES[j]:16s} {col.min():8.2f} {col.max():8.2f} "
          f"{col.mean():8.3f} {col.std():8.3f} {col.max()-col.min():8.2f}")
print(f"{'price (lakh)':16s} {y_train.min():8.2f} {y_train.max():8.2f} "
      f"{y_train.mean():8.3f} {y_train.std():8.3f} {y_train.max()-y_train.min():8.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for j, ax in enumerate(axes):
    ax.scatter(X_train[:, j], y_train, s=28, c="#1B7F79", alpha=0.85)
    ax.set_xlabel(FEATURES[j])
axes[0].set_ylabel("price (lakh BDT)")
fig.suptitle("Price against each feature separately", y=1.02)
plt.tight_layout()
plt.show()

Size and bedrooms rise with price, age falls, floor is nearly flat. But each panel shows one
feature *ignoring* the other three, and those are correlated &mdash; bigger flats have more
bedrooms. A single panel cannot tell you a feature's contribution once the others are
accounted for. That is precisely what fitting all four weights at once does.

---

## 2. The model

$$f_{\mathbf{w},b}(\mathbf{x}) = w_1x_1 + w_2x_2 + \dots + w_nx_n + b
= \mathbf{w}\cdot\mathbf{x} + b$$

Start with one example, three ways, as in the lecture.

In [ ]:
w_guess = np.array([20.0, 2.0, 1.0, -0.3])
b_guess = 10.0
x_i = X_train[0]

# 1. Written out -- fine for n = 4, unwritable for n = 100.
f_written = (w_guess[0]*x_i[0] + w_guess[1]*x_i[1]
             + w_guess[2]*x_i[2] + w_guess[3]*x_i[3] + b_guess)

# 2. A loop.
f_loop = 0.0
for j in range(n):
    f_loop += w_guess[j] * x_i[j]
f_loop += b_guess

# 3. Vectorised -- the only version to write.
f_vec = np.dot(w_guess, x_i) + b_guess

print(f"written out : {f_written:.4f}")
print(f"loop        : {f_loop:.4f}")
print(f"vectorised  : {f_vec:.4f}")
print(f"\nactual price y^(0) = {y_train[0]:.1f}  ->  residual {f_vec - y_train[0]:+.4f}")

assert np.allclose([f_written, f_loop], f_vec)

### All $m$ predictions at once

`X @ w` multiplies the `(m, n)` matrix by the `(n,)` vector and returns `(m,)` &mdash; one
prediction per example. Adding the scalar `b` broadcasts across all of them.

In [ ]:
def predict(X, w, b):
    """
    Linear model prediction for every example.

    Args:
        X (ndarray (m, n)): feature matrix, one row per example
        w (ndarray (n,))  : weights
        b (scalar)        : bias

    Returns:
        f_wb (ndarray (m,)): one prediction per example
    """
    assert X.ndim == 2, f"X must be 2-D, got shape {X.shape}"
    assert w.shape == (X.shape[1],), f"w must have shape ({X.shape[1]},), got {w.shape}"
    return X @ w + b


preds = predict(X_train, w_guess, b_guess)
print("predictions shape:", preds.shape)
print("first five       :", preds[:5])
print("actual first five:", y_train[:5])

# The loop version, to prove the one-liner is doing what we think.
preds_loop = np.array([np.dot(w_guess, X_train[i]) + b_guess for i in range(m)])
assert np.allclose(preds, preds_loop)
print("\nMatrix form agrees with the per-example loop.")

---

## 3. The cost

$$J(\mathbf{w},b) = \frac{1}{2m}\sum_{i=1}^{m}
\bigl(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\bigr)^2$$

Identical in form to Week 1. Only $f$ changed.

In [ ]:
def compute_cost(X, y, w, b):
    """Squared-error cost for the multi-feature linear model."""
    err = predict(X, w, b) - y                # shape (m,)
    return float(np.sum(err ** 2) / (2 * X.shape[0]))


def compute_cost_loop(X, y, w, b):
    """The same cost, written with an explicit loop over examples."""
    m = X.shape[0]
    total = 0.0
    for i in range(m):
        f_i = np.dot(w, X[i]) + b
        total += (f_i - y[i]) ** 2
    return total / (2 * m)


print(f"J(w_guess, b_guess) = {compute_cost(X_train, y_train, w_guess, b_guess):.4f}")
print(f"J(0, 0)             = {compute_cost(X_train, y_train, np.zeros(n), 0.0):.4f}")

assert np.allclose(compute_cost(X_train, y_train, w_guess, b_guess),
                   compute_cost_loop(X_train, y_train, w_guess, b_guess))
print("\nLoop and vectorised costs agree.")

$J(\mathbf{0}, 0)$ is large because a model predicting zero for every flat is wrong by the
whole price. It is the baseline any trained model must beat by a wide margin.

---

## 4. The gradient

$$\frac{\partial J}{\partial w_j}
= \frac{1}{m}\sum_{i=1}^{m}\bigl(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\bigr)x^{(i)}_j,
\qquad
\frac{\partial J}{\partial b}
= \frac{1}{m}\sum_{i=1}^{m}\bigl(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\bigr)$$

One derivative per weight, all sharing the same residual vector. Write the double loop first
so the formula is visible, then the one-liner.

In [ ]:
def compute_gradient_loop(X, y, w, b):
    """Gradient of the squared-error cost, with explicit loops over examples and features."""
    m, n = X.shape
    dj_dw = np.zeros(n)
    dj_db = 0.0
    for i in range(m):
        err = (np.dot(w, X[i]) + b) - y[i]      # a scalar, shared by every j
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        dj_db += err
    return dj_dw / m, dj_db / m


def compute_gradient(X, y, w, b):
    """
    Gradient of the squared-error cost (vectorised).

    Returns:
        dj_dw (ndarray (n,)): one partial derivative per weight
        dj_db (float)       : partial derivative with respect to b
    """
    m = X.shape[0]
    err = predict(X, w, b) - y                  # (m,)
    dj_dw = X.T @ err / m                       # (n, m) @ (m,) -> (n,)
    dj_db = float(np.sum(err) / m)
    return dj_dw, dj_db


dw_l, db_l = compute_gradient_loop(X_train, y_train, w_guess, b_guess)
dw_v, db_v = compute_gradient(X_train, y_train, w_guess, b_guess)

print("loop       dj_dw =", dw_l, " dj_db =", round(db_l, 4))
print("vectorised dj_dw =", dw_v, " dj_db =", round(db_v, 4))

assert np.allclose(dw_l, dw_v) and np.allclose(db_l, db_v)
print("\nThe two implementations agree.")

> **Why `X.T @ err` is the weight gradient.** Row $j$ of $X^{\top}$ is feature $j$ across every
> example. Dotting it with the residual vector gives
> $\sum_i (f^{(i)} - y^{(i)})x^{(i)}_j$ &mdash; exactly $m\,\partial J/\partial w_j$. The whole
> gradient is one matrix&ndash;vector product.

### Check the gradient before trusting it

The same central-difference check as Week 1, now applied to each of the $n+1$ parameters in
turn. A wrong gradient is the bug that silently produces a badly fitted model.

In [ ]:
def gradient_check(X, y, w, b, eps=1e-6):
    """Compare the analytic gradient with central finite differences, parameter by parameter."""
    ana_dw, ana_db = compute_gradient(X, y, w, b)

    num_dw = np.zeros_like(w)
    for j in range(w.shape[0]):
        wp = w.copy(); wp[j] += eps
        wm = w.copy(); wm[j] -= eps
        num_dw[j] = (compute_cost(X, y, wp, b) - compute_cost(X, y, wm, b)) / (2 * eps)
    num_db = (compute_cost(X, y, w, b + eps) - compute_cost(X, y, w, b - eps)) / (2 * eps)

    return (ana_dw, num_dw), (ana_db, num_db)


for w_t, b_t in [(np.zeros(n), 0.0), (w_guess, b_guess), (np.array([-5., 30., 0., 2.]), -40.0)]:
    (a_w, nu_w), (a_b, nu_b) = gradient_check(X_train, y_train, w_t, b_t)
    rel = np.max(np.abs(a_w - nu_w) / (np.abs(a_w) + np.abs(nu_w) + 1e-12))
    print(f"at w={np.round(w_t, 1)}, b={b_t:6.1f}   max relative error = {rel:.2e}")
    assert np.allclose(a_w, nu_w, rtol=1e-5, atol=1e-4)
    assert np.isclose(a_b, nu_b, rtol=1e-5, atol=1e-4)

print("\nGradient check passed at every test point.")

---

## 5. Gradient descent for $n$ features

$$w_j := w_j - \alpha\frac{\partial J}{\partial w_j}\ \ (j = 1,\dots,n),
\qquad b := b - \alpha\frac{\partial J}{\partial b}$$

**updated simultaneously** &mdash; which now means all $n+1$ parameters. The code below computes
the entire gradient from the current $(\mathbf{w}, b)$ before either assignment happens.

In [ ]:
def gradient_descent(X, y, alpha, num_iters, w_init=None, b_init=0.0, record_every=1):
    """
    Batch gradient descent for multiple linear regression.

    Returns:
        w (ndarray (n,)), b (float), history (dict with 'cost')
    """
    w = np.zeros(X.shape[1]) if w_init is None else np.array(w_init, dtype=float)
    b = float(b_init)
    history = {"cost": []}

    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)   # both, from the current parameters
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i % record_every == 0 or i == num_iters - 1:
            history["cost"].append(compute_cost(X, y, w, b))

    return w, b, history

### Choosing $\alpha$ is already a problem

Try the Week 1 value of `0.1` on this data set.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")               # overflow is the point here
    w_bad, b_bad, h_bad = gradient_descent(X_train, y_train, alpha=0.1, num_iters=12)

print("cost over the first twelve iterations with alpha = 0.1:")
for i, c in enumerate(h_bad["cost"]):
    print(f"  iter {i:2d}:  J = {c:.4e}")

growth = h_bad["cost"][-1] / h_bad["cost"][0]
print(f"\ncost grew by a factor of {growth:.2e} in eleven steps"
      f"  (about {growth ** (1/11):.1e} per iteration)")

assert h_bad["cost"][-1] > 1e40, "expected explosive divergence"
print("Diverged. On the raw features the usable alpha is far smaller.")

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results = {}
    for a in [1e-4, 5e-4, 1e-3, 1.5e-3, 2e-3]:
        _, _, h = gradient_descent(X_train, y_train, alpha=a, num_iters=2000)
        results[a] = h["cost"][-1]

print("J after 2000 iterations:")
for a, c in results.items():
    print(f"  alpha = {a:<8g} J = {('%.4f' % c) if np.isfinite(c) else 'diverged'}")

So the whole usable range of $\alpha$ is squeezed below about $0.0018$, and even the best of
these is still far from converged after 2000 iterations. Run the best one much longer.

In [ ]:
w_gd, b_gd, hist = gradient_descent(X_train, y_train, alpha=0.0015,
                                    num_iters=50_000, record_every=50)

print(f"after 50,000 iterations:")
print(f"  w = {w_gd}")
print(f"  b = {b_gd:.4f}")
print(f"  J = {compute_cost(X_train, y_train, w_gd, b_gd):.6f}")

### An independent check on the answer

Linear regression has a closed form, so we can confirm where the minimum actually is.
`np.linalg.lstsq` solves it directly &mdash; this is the lecture's *test oracle*, not a
replacement for gradient descent.

In [ ]:
A = np.column_stack([X_train, np.ones(m)])          # design matrix, last column of ones for b
coef, *_ = np.linalg.lstsq(A, y_train, rcond=None)
w_exact, b_exact = coef[:n], coef[n]
J_min = compute_cost(X_train, y_train, w_exact, b_exact)

print(f"{'':16s} {'gradient descent':>18} {'closed form':>14}")
print("-" * 52)
for j in range(n):
    print(f"{FEATURES[j]:16s} {w_gd[j]:18.4f} {w_exact[j]:14.4f}")
print(f"{'bias b':16s} {b_gd:18.4f} {b_exact:14.4f}")
print(f"\nJ:  gradient descent {compute_cost(X_train, y_train, w_gd, b_gd):.6f}"
      f"    minimum {J_min:.6f}")

assert np.allclose(w_gd, w_exact, atol=0.2), "gradient descent has not converged"
print("\nClose, but not equal -- after 50,000 iterations it is still crawling.")

In [ ]:
plt.figure(figsize=(11, 3.8))

plt.subplot(1, 2, 1)
iters = np.arange(len(hist["cost"])) * 50
plt.plot(iters, hist["cost"], c="#123B63", lw=2)
plt.axhline(J_min, ls="--", c="#C97B17", lw=1.4, label=f"minimum J = {J_min:.3f}")
plt.xlabel("iteration"); plt.ylabel("J(w, b)")
plt.title("alpha = 0.0015, 50,000 iterations")
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
plt.plot(iters, np.array(hist["cost"]) - J_min, c="#9B2C4B", lw=2)
plt.yscale("log")
plt.xlabel("iteration"); plt.ylabel("J - J_min   (log scale)")
plt.title("Distance from the minimum")

plt.tight_layout()
plt.show()

The right-hand panel is the honest picture: the excess cost falls, but slowly and steadily,
with no sign of the sharp collapse a well-conditioned problem gives. Nothing here is broken.
The gradient is verified, the update is simultaneous, $J$ decreases on every iteration &mdash;
and the algorithm is still uselessly slow.

The cause is in the ranges you printed in section 1: age spans 2 to 59 while size spans 0.62
to 2.99. Part 3 fixes it in three lines.

### Checkpoint 1

Confirm that $J$ decreased on **every** recorded iteration of the run above, using
`np.diff`, and report the number of iterations where it did not. Then compute the gradient at
the closed-form solution `(w_exact, b_exact)` and report the largest absolute component.

*Expected:* zero increases, and a gradient whose largest component is below `1e-10`.

In [ ]:
# Your code here

### Checkpoint 2

Use the closed-form model to predict the price of a flat with

| size | bedrooms | floor | age |
|---|---|---|---|
| 1.80 | 3 | 5 | 12 |

Then print each of the five contributions ($w_1x_1, \dots, w_4x_4$ and $b$) separately, as
in Part 1, and say in one sentence which feature contributes most to the prediction.

*Expected:* the predicted price is approximately `56.21` lakh BDT.

In [ ]:
# Your code here

---

## 6. Recap

- A multi-feature training set is `X` of shape `(m, n)`: row $i$ is $\mathbf{x}^{(i)}$,
  column $j$ is feature $j$, `X[i, j]` is $x^{(i)}_j$.
- The model is `X @ w + b`; the cost is unchanged in form from Week 1.
- The gradient is `X.T @ err / m` for the weights and `err.mean()` for the bias &mdash; one
  matrix&ndash;vector product, not a double loop.
- Verify every gradient against finite differences before training with it.
- `np.linalg.lstsq` gives the exact minimum for *this* model only; use it as a test oracle.
- Features on wildly different scales force $\alpha$ to be tiny and make convergence
  unusable. That is a property of the data, not a bug in the code.

### Exercises to hand in

1. Fit the model on the first two features only (`X_train[:, :2]`), then on all four. Report
   $J_{\min}$ in each case, and explain why adding features can never *increase* the minimum
   training cost.
2. The true generating rule in `load_flats` uses `w = [22, 2.5, 0.9, -0.35]`, `b = 8`.
   Compare it with `w_exact`, `b_exact`. Which weight is recovered worst? Investigate whether
   the correlation between size and bedrooms explains it, by computing
   `np.corrcoef(X_train[:, 0], X_train[:, 1])`.
3. Duplicate a column (`X_dup = np.column_stack([X_train, X_train[:, 0]])`) and fit again.
   Report what `np.linalg.lstsq` returns for the two identical columns and what the minimum
   cost becomes. Explain, using the lecture's remark about $X^{\top}X$ being singular.
4. Time `compute_gradient_loop` against `compute_gradient` on this data set, then estimate
   the time each would take for $m = 10^5$, $n = 50$, over 10,000 iterations.
5. Add an `assert` to `gradient_descent` that raises as soon as $J$ increases, reporting the
   iteration number and the value of $\alpha$. Verify it fires for `alpha = 0.1` and stays
   silent for `alpha = 0.0015`. Explain why failing loudly is preferable to returning `nan`.

### Next

**Part 3 &mdash; Feature Scaling and the Learning Rate:** why this run was so slow, the three
scaling methods, and a systematic way to choose $\alpha$.

**Reading:** James et al., *ISL* 2e, &sect;3.2; Bishop, *PRML*, &sect;3.1&ndash;3.1.1.